In [1]:
import json
import pandas as pd
from tqdm import tqdm
import re
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Dict, Set, List
from collections import defaultdict


## Drugs @ FDA

In [2]:
fda_drugs_file = "/shares/animalwelfare.crs.uzh/fda_files/2025_11_11/drug-drugsfda-0001-of-0001.json"

In [3]:

# Load the FDA JSON file
with open(fda_drugs_file, "r") as f:
    data = json.load(f)

# Initialize list for filtered results
orig_records = []

# Loop over results
for result in data.get("results", []):
    submissions = result.get("submissions", [])
    
    # Find the ORIG submission(s)
    orig_subs = [s for s in submissions if s.get("submission_type") == "ORIG"]
    if not orig_subs:
        continue

    orig = orig_subs[0]
    submission_date = orig.get("submission_status_date")
    year = submission_date[:4] if submission_date else None
    # Extract base info
    record = {
        "application_number": result.get("application_number"),
        "sponsor_name": result.get("sponsor_name"),
        "submission_type": orig.get("submission_type"),
        "submission_status": orig.get("submission_status"),
        "submission_status_date": submission_date,
        "submission_class_code_description": orig.get("submission_class_code_description"),
        "year": year
    }

    # Extract first product if exists
    products = result.get("products", [])
    if products:
        p = products[0]
        record.update({
            "brand_name": p.get("brand_name"),
            "dosage_form": p.get("dosage_form"),
            "route": p.get("route"),
            "marketing_status": p.get("marketing_status"),
            "active_ingredients": ", ".join(
                f"{a.get('name')}" for a in p.get("active_ingredients", [])
            )
        })

    # If openfda section exists, flatten a few key fields
    openfda = result.get("openfda", {})
    if openfda:
        record.update({
            "openfda_brand_name": ", ".join(openfda.get("brand_name", [])),
            "openfda_generic_name": ", ".join(openfda.get("generic_name", [])),
            "openfda_route": ", ".join(openfda.get("route", [])),
            "openfda_substance_name": ", ".join(openfda.get("substance_name", []))
        })

    orig_records.append(record)

# Convert to DataFrame for easier viewing/analysis
df_orig = pd.DataFrame(orig_records)

In [4]:
df_orig.shape

(25712, 16)

In [5]:
df_orig.submission_status.value_counts()

submission_status
AP    24636
TA     1075
Name: count, dtype: int64

In [6]:
# Show summary
print("Found", len(df_orig), "ORIG applications")
df_orig.head()

Found 25712 ORIG applications


,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,marketing_status,active_ingredients,openfda_brand_name,openfda_generic_name,openfda_route,openfda_substance_name
0,ANDA076194,WATSON LABS,ORIG,AP,20020701,None,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,Prescription,"HYDROCHLOROTHIAZIDE, LISINOPRIL",LISINOPRIL AND HYDROCHLOROTHIAZIDE,LISINOPRIL AND HYDROCHLOROTHIAZIDE,ORAL,"HYDROCHLOROTHIAZIDE, LISINOPRIL"
1,ANDA076206,ROCKWELL MEDCL,ORIG,AP,20030917,None,2003,CALCITRIOL,INJECTABLE,INJECTION,Discontinued,CALCITRIOL,NaN,NaN,NaN,NaN
2,ANDA076212,APOTEX,ORIG,AP,20040616,None,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,Discontinued,"CARBIDOPA, LEVODOPA",NaN,NaN,NaN,NaN
3,ANDA076215,FOUGERA PHARMS,ORIG,AP,20031209,None,2003,BETAMETHASONE DIPROPIONATE,"CREAM, AUGMENTED",TOPICAL,Prescription,BETAMETHASONE DIPROPIONATE,NaN,NaN,NaN,NaN
4,ANDA076224,PHARMOBEDIENT,ORIG,AP,20030509,None,2003,FLUTAMIDE,CAPSULE,ORAL,Discontinued,FLUTAMIDE,NaN,NaN,NaN,NaN


In [7]:
df_orig[df_orig['active_ingredients']=="CLADRIBINE"]

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,marketing_status,active_ingredients,openfda_brand_name,openfda_generic_name,openfda_route,openfda_substance_name
2062,NDA022561,EMD SERONO INC,ORIG,AP,20190329,Type 3 - New Dosage Form,2019,MAVENCLAD,TABLET,ORAL,Prescription,CLADRIBINE,MAVENCLAD,CLADRIBINE,ORAL,CLADRIBINE
8585,ANDA200510,PHARMOBEDIENT,ORIG,AP,20111006,Not Applicable,2011,CLADRIBINE,INJECTABLE,INJECTION,Discontinued,CLADRIBINE,NaN,NaN,NaN,NaN
11798,NDA020229,JANSSEN PHARMS,ORIG,AP,19930226,Type 1 - New Molecular Entity,1993,LEUSTATIN,INJECTABLE,INJECTION,Discontinued,CLADRIBINE,NaN,NaN,NaN,NaN
12849,ANDA075405,HIKMA,ORIG,AP,20000228,None,2000,CLADRIBINE,INJECTABLE,INJECTION,Prescription,CLADRIBINE,CLADRIBINE,CLADRIBINE,INTRAVENOUS,CLADRIBINE
20606,ANDA076571,FRESENIUS KABI USA,ORIG,AP,20040422,None,2004,CLADRIBINE,INJECTABLE,INJECTION,Prescription,CLADRIBINE,CLADRIBINE,CLADRIBINE,INTRAVENOUS,CLADRIBINE
24464,ANDA210856,HISUN PHARM HANGZHOU,ORIG,AP,20191125,None,2019,CLADRIBINE,INJECTABLE,INJECTION,Prescription,CLADRIBINE,CLADRIBINE,CLADRIBINE,INTRAVENOUS,CLADRIBINE


## Label

In [8]:
file_nr = "01"
drug_label_file = f"/shares/animalwelfare.crs.uzh/fda_files/2025_11_11/drug-label-00{file_nr}-of-0013.json"

# Load your drug label JSON (replace filename with yours)
with open(drug_label_file, "r") as f:
    data = json.load(f)

results = data.get("results", [])

records = []

for result in tqdm(results, desc="Processing FDA drug labels", unit="label"):
    openfda = result.get("openfda", {})
    application_numbers = openfda.get("application_number", [])

    # ✅ Skip if no application_number
    if not application_numbers:
        continue

    clinical_text = " ".join(result.get("clinical_studies", []))
    nct_ids = re.findall(r"NCT\s*[-:]?\s*\d{6,8}", clinical_text, flags=re.IGNORECASE)

    # --- Extract only the first sentence from indications_and_usage
    indications_text = " ".join(result.get("indications_and_usage", []))
    match = re.match(r"^(.*?)(?<!\d)\.\s", indications_text)
    first_sentence = match.group(1).strip() if match else indications_text.strip()

    record = {
        "application_number": ", ".join(openfda.get("application_number", [])),
        "label_brand_name": ", ".join(openfda.get("brand_name", [])),
        "label_generic_name": ", ".join(openfda.get("generic_name", [])),
        "label_manufacturer_name": ", ".join(openfda.get("manufacturer_name", [])),
        "label_substance_name": ", ".join(openfda.get("substance_name", [])),
        "indications_first_sent": first_sentence,
        "indications_and_usage": indications_text,
        "clinical_studies": nct_ids,
    }

    records.append(record)

# Convert to DataFrame
df_labels = pd.DataFrame(records)
df_orig = df_orig.merge(df_labels, on="application_number", how="left")

Processing FDA drug labels: 100%|██████████| 20000/20000 [00:00<00:00, 151956.25label/s]


In [9]:
df_labels.head()

,application_number,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
0,ANDA209057,Vardenafil Hydrochloride,VARDENAFIL HYDROCHLORIDE,Bryant Ranch Prepack,VARDENAFIL HYDROCHLORIDE,1 INDICATIONS AND USAGE Vardenafil hydrochlori...,1 INDICATIONS AND USAGE Vardenafil hydrochlori...,[]
1,ANDA211610,ACETAMINOPHEN AND CODEINE PHOSPHATE,ACETAMINOPHEN AND CODEINE PHOSPHATE,Eywa Pharma Inc,"ACETAMINOPHEN, CODEINE PHOSPHATE",INDICATIONS AND USAGE Acetaminophen and codein...,INDICATIONS AND USAGE Acetaminophen and codein...,[]
2,ANDA214403,"Dextroamphetamine Saccharate, Amphetamine Aspa...","DEXTROAMPHETAMINE SULFATE, DEXTROAMPHETAMINE S...","Golden State Medical Supply, Inc.","AMPHETAMINE ASPARTATE MONOHYDRATE, AMPHETAMINE...",1 INDICATIONS AND USAGE Dextroamphetamine sacc...,1 INDICATIONS AND USAGE Dextroamphetamine sacc...,[]
3,ANDA211355,OMEGA-3-ACID ETHYL ESTERS,OMEGA-3-ACID ETHYL ESTERS,Procaps S A,OMEGA-3-ACID ETHYL ESTERS,1 INDICATION S AND USAGE Omega-3-Acid Ethyl Es...,1 INDICATION S AND USAGE Omega-3-Acid Ethyl Es...,[]
4,ANDA076674,LISINOPRIL and HYDROCHLOROTHIAZIDE,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"GSMS, Incorporated","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[]


In [10]:
contains_melatonin = (
    df_labels
    .astype(str)
    .apply(lambda col: col.str.contains("melatonin", case=False, na=False))
    .any()
    .any()
)

contains_melatonin


False

In [11]:
import json
import re
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# --- only these columns will be written into df_orig
label_cols = [
    "label_brand_name",
    "label_generic_name",
    "label_manufacturer_name",
    "label_substance_name",
    "indications_first_sent",
    "indications_and_usage",
    "clinical_studies",
]

# ensure df_orig has target columns (so .update works cleanly)
for col in label_cols:
    if col not in df_orig.columns:
        df_orig[col] = pd.NA

# regex: first sentence without splitting on numbered list markers like "1."
FIRST_SENTENCE_RE = re.compile(r"^(.*?)(?<!\d)\.\s")

def first_sentence(text: str) -> str:
    if not text:
        return ""
    m = FIRST_SENTENCE_RE.match(text)
    return (m.group(1) if m else text).strip()

# --- iterate files 01..13, process and merge immediately
base_dir = Path("/shares/animalwelfare.crs.uzh/fda_files/2025_11_11")

for i in tqdm(range(1, 14), desc="Processing label files", unit="file"):
    file_nr = f"{i:02d}"
    drug_label_file = base_dir / f"drug-label-00{file_nr}-of-0013.json"

    with open(drug_label_file, "r") as f:
        data = json.load(f)

    results = data.get("results", [])

    # build minimal records for THIS file only
    records = []
    for result in tqdm(results, desc=f"Parsing {drug_label_file.name}", unit="label", leave=False):
        openfda = result.get("openfda", {}) or {}
        application_numbers = openfda.get("application_number", []) or []

        # Skip if no application_number
        if not application_numbers:
            continue

        # clinical trials: tolerant NCT regex (as you wrote)
        clinical_text = " ".join(result.get("clinical_studies", []) or [])
        nct_ids = re.findall(r"NCT\s*[-:]?\s*\d{6,8}", clinical_text, flags=re.IGNORECASE)

        # indications: first sentence only (don’t split on "1.")
        indications_text = " ".join(result.get("indications_and_usage", []) or [])
        match = re.match(r"^(.*?)(?<!\d)\.\s", indications_text)
        first_sent = match.group(1).strip() if match else indications_text.strip()

        # one row per application_number (clean merge key)
        for app_no in application_numbers:
            records.append({
                "application_number": app_no,
                "label_brand_name": ", ".join(openfda.get("brand_name", []) or []),
                "label_generic_name": ", ".join(openfda.get("generic_name", []) or []),
                "label_manufacturer_name": ", ".join(openfda.get("manufacturer_name", []) or []),
                "label_substance_name": ", ".join(openfda.get("substance_name", []) or []),
                "indications_first_sent": first_sent,
                "indications_and_usage": indications_text,
                "clinical_studies": nct_ids,  # keep as list
            })

    # turn this file's records into a df, dedupe by application_number to avoid row explosion
    df_labels_file = pd.DataFrame(records)
    if not df_labels_file.empty:
        df_labels_file = df_labels_file.drop_duplicates(subset=["application_number"], keep="first")
        # in-place update (no _x/_y columns, no row duplication)
        df_orig = df_orig.set_index("application_number")
        df_orig.update(df_labels_file.set_index("application_number")[label_cols])
        df_orig = df_orig.reset_index()

print("✅ Finished merging each label file into df_orig (on the fly).")


Processing label files: 100%|██████████| 13/13 [00:55<00:00,  4.24s/file]                       

✅ Finished merging each label file into df_orig (on the fly).


In [12]:
df_orig

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,openfda_generic_name,openfda_route,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
0,ANDA076194,WATSON LABS,ORIG,AP,20020701,None,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,LISINOPRIL AND HYDROCHLOROTHIAZIDE,ORAL,"HYDROCHLOROTHIAZIDE, LISINOPRIL",Lisinopril and Hydrochlorothiazide,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"Actavis Pharma, Inc.","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[]
1,ANDA076206,ROCKWELL MEDCL,ORIG,AP,20030917,None,2003,CALCITRIOL,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ANDA076212,APOTEX,ORIG,AP,20040616,None,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ANDA076215,FOUGERA PHARMS,ORIG,AP,20031209,None,2003,BETAMETHASONE DIPROPIONATE,"CREAM, AUGMENTED",TOPICAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ANDA076224,PHARMOBEDIENT,ORIG,AP,20030509,None,2003,FLUTAMIDE,CAPSULE,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26301,ANDA076154,MYLAN,ORIG,AP,20030820,None,2003,LORATADINE,TABLET,ORAL,...,LORATADINE,ORAL,LORATADINE,Loratadine,LORATADINE,Mylan Pharmaceuticals Inc.,LORATADINE,Uses temporarily relieves these symptoms due t...,Uses temporarily relieves these symptoms due t...,[]
26302,ANDA076155,APOTEX INC,ORIG,AP,20030418,None,2003,IPRATROPIUM BROMIDE,"SPRAY, METERED",NASAL,...,IPRATROPIUM BROMIDE,NASAL,IPRATROPIUM BROMIDE,Ipratropium Bromide,IPRATROPIUM BROMIDE,Apotex Corp.,IPRATROPIUM BROMIDE,INDICATIONS AND USAGE Ipratropium bromide nasa...,INDICATIONS AND USAGE Ipratropium bromide nasa...,[]
26303,ANDA076162,WATSON LABS TEVA,ORIG,AP,20041014,None,2004,CARBOPLATIN,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26304,ANDA076163,DR REDDYS,ORIG,AP,20030905,None,2003,AMIODARONE HYDROCHLORIDE,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_orig[df_orig['active_ingredients']=="CLADRIBINE"]

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,openfda_generic_name,openfda_route,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
2116,NDA022561,EMD SERONO INC,ORIG,AP,20190329,Type 3 - New Dosage Form,2019,MAVENCLAD,TABLET,ORAL,...,CLADRIBINE,ORAL,CLADRIBINE,Mavenclad,CLADRIBINE,"EMD Serono, Inc.",CLADRIBINE,1 INDICATIONS AND USAGE MAVENCLAD is indicated...,1 INDICATIONS AND USAGE MAVENCLAD is indicated...,[NCT00213135]
8756,ANDA200510,PHARMOBEDIENT,ORIG,AP,20111006,Not Applicable,2011,CLADRIBINE,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12041,NDA020229,JANSSEN PHARMS,ORIG,AP,19930226,Type 1 - New Molecular Entity,1993,LEUSTATIN,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13100,ANDA075405,HIKMA,ORIG,AP,20000228,None,2000,CLADRIBINE,INJECTABLE,INJECTION,...,CLADRIBINE,INTRAVENOUS,CLADRIBINE,Cladribine,CLADRIBINE,Hikma Pharmaceuticals USA Inc.,CLADRIBINE,"INDICATIONS AND USAGE Cladribine Injection, US...","INDICATIONS AND USAGE Cladribine Injection, US...",[]
21084,ANDA076571,FRESENIUS KABI USA,ORIG,AP,20040422,None,2004,CLADRIBINE,INJECTABLE,INJECTION,...,CLADRIBINE,INTRAVENOUS,CLADRIBINE,Cladribine,CLADRIBINE,"Fresenius Kabi USA, LLC",CLADRIBINE,"INDICATIONS AND USAGE: Cladribine Injection, U...","INDICATIONS AND USAGE: Cladribine Injection, U...",[]
25046,ANDA210856,HISUN PHARM HANGZHOU,ORIG,AP,20191125,None,2019,CLADRIBINE,INJECTABLE,INJECTION,...,CLADRIBINE,INTRAVENOUS,CLADRIBINE,Cladribine,CLADRIBINE,"Hisun Pharmaceuticals USA, Inc.",CLADRIBINE,"INDICATIONS FOR USE Cladribine Injection, USP ...","INDICATIONS FOR USE Cladribine Injection, USP ...",[]


In [14]:
df_orig.to_csv("out/FDA_drugs_labels_full.csv",index=False)
df_orig.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full.csv",index=False)

### embed drug terms (for merge with target DS and parsing of indications)

In [15]:
df_orig = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full.csv")

In [16]:
melatonin_rows = df_orig[
    df_orig
    .astype(str)
    .apply(lambda col: col.str.contains("amisulpride", case=False, na=False))
    .any(axis=1)
]
melatonin_rows

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,openfda_generic_name,openfda_route,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
9302,NDA209510,ACACIA,ORIG,AP,20200226,Type 1 - New Molecular Entity,2020,BARHEMSYS,SOLUTION,INTRAVENOUS,...,AMISULPRIDE,INTRAVENOUS,AMISULPRIDE,Barhemsys,AMISULPRIDE,Acacia Pharma Ltd,AMISULPRIDE,1 INDICATIONS AND USAGE BARHEMSYS ® is indicat...,1 INDICATIONS AND USAGE BARHEMSYS ® is indicat...,"['NCT01991860', 'NCT02337062', 'NCT02449291', ..."


### select approvals

In [17]:
df_orig = df_orig[df_orig['submission_status']=="AP"]
df_orig.shape

(25230, 23)

In [18]:
cols = ["active_ingredients", "label_substance_name", "brand_name"]

df_orig["active_ingredients_substance_brand"] = (
    df_orig[cols]
    .fillna("")
    .apply(
        lambda row: ", ".join(
            dict.fromkeys(  # preserves order while removing duplicates
                v.strip() for v in row if v.strip()
            )
        ),
        axis=1
    )
)


In [19]:
terms = df_orig[['active_ingredients','label_substance_name','active_ingredients_substance_brand']]
terms[terms['label_substance_name']=="CARIPRAZINE"]

,active_ingredients,label_substance_name,active_ingredients_substance_brand
21966,CARIPRAZINE HYDROCHLORIDE,CARIPRAZINE,"CARIPRAZINE HYDROCHLORIDE, CARIPRAZINE, VRAYLAR"


In [20]:
df_orig = df_orig[
    df_orig["active_ingredients_substance_brand"].notna() &
    (df_orig["active_ingredients_substance_brand"].astype(str).str.strip() != "")
]

df_orig.shape

(25216, 24)

In [21]:
df_orig["active_ingredients_split"] = (
    df_orig["active_ingredients_substance_brand"]
    .astype(str)
    .str.split(r"(?i),\s*|\|\||\s*\band\b\s*", regex=True)
    .apply(
        lambda lst: list(
            dict.fromkeys(  # de-dupe, keep order
                s.strip() for s in lst if s and s.strip()
            )
        )
    )
)

# Explode into separate rows
df_orig = df_orig.explode("active_ingredients_split", ignore_index=True)

df_orig.shape

(38900, 25)

In [22]:
df_orig.head()

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies,active_ingredients_substance_brand,active_ingredients_split
0,ANDA076194,WATSON LABS,ORIG,AP,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,"HYDROCHLOROTHIAZIDE, LISINOPRIL",Lisinopril and Hydrochlorothiazide,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"Actavis Pharma, Inc.","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[],"HYDROCHLOROTHIAZIDE, LISINOPRIL, LISINOPRIL AN...",HYDROCHLOROTHIAZIDE
1,ANDA076194,WATSON LABS,ORIG,AP,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,"HYDROCHLOROTHIAZIDE, LISINOPRIL",Lisinopril and Hydrochlorothiazide,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"Actavis Pharma, Inc.","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[],"HYDROCHLOROTHIAZIDE, LISINOPRIL, LISINOPRIL AN...",LISINOPRIL
2,ANDA076206,ROCKWELL MEDCL,ORIG,AP,20030917,NaN,2003,CALCITRIOL,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CALCITRIOL,CALCITRIOL
3,ANDA076212,APOTEX,ORIG,AP,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"CARBIDOPA, LEVODOPA, CARBIDOPA AND LEVODOPA",CARBIDOPA
4,ANDA076212,APOTEX,ORIG,AP,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"CARBIDOPA, LEVODOPA, CARBIDOPA AND LEVODOPA",LEVODOPA


In [23]:
df_orig[df_orig["active_ingredients_split"].str.contains("cariprazine", case=False, na=False)]


,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies,active_ingredients_substance_brand,active_ingredients_split
32480,NDA204370,ABBVIE,ORIG,AP,20150917,Type 1 - New Molecular Entity,2015,VRAYLAR,CAPSULE,ORAL,...,CARIPRAZINE,Vraylar,CARIPRAZINE,"Allergan, Inc.",CARIPRAZINE,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,[],"CARIPRAZINE HYDROCHLORIDE, CARIPRAZINE, VRAYLAR",CARIPRAZINE HYDROCHLORIDE
32481,NDA204370,ABBVIE,ORIG,AP,20150917,Type 1 - New Molecular Entity,2015,VRAYLAR,CAPSULE,ORAL,...,CARIPRAZINE,Vraylar,CARIPRAZINE,"Allergan, Inc.",CARIPRAZINE,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,[],"CARIPRAZINE HYDROCHLORIDE, CARIPRAZINE, VRAYLAR",CARIPRAZINE
37289,ANDA213932,SUN PHARM,ORIG,AP,20220930,NaN,2022,CARIPRAZINE HYDROCHLORIDE,CAPSULE,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CARIPRAZINE HYDROCHLORIDE,CARIPRAZINE HYDROCHLORIDE
37291,ANDA213984,ZYDUS,ORIG,AP,20220909,NaN,2022,CARIPRAZINE HYDROCHLORIDE,CAPSULE,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CARIPRAZINE HYDROCHLORIDE,CARIPRAZINE HYDROCHLORIDE


In [24]:
df_orig.to_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full_AP.csv",index=False)

### map flat incredients to ontology

In [25]:
import sys
sys.path.append("../04_normalization")   # adjust path to your real folder
from neural_based_nen import main

In [26]:
data_dir =  "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/"
mapping_type = "drug"
col_to_map = "active_ingredients_split"
input_file = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full_AP.csv"
output_file = "/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full_AP_mapped_active_ingredients.csv"

main(mapping_type, col_to_map, data_dir, input_file, output_file, stats_dir=None)

Input file: /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full_AP.csv


/home/sdonev/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Using terminology: umls
Using distance threshold: 8.2
Starting normalization for: DRUG with cdist 8.2
Loading embeddings from /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/umls/embeddings with prefix UMLS_emb...
Loading term-id pairs from /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/umls/umls_term_id_pairs_combined.json...
Found COMBINED embeddings file, loading that one...
Loaded embeddings: torch.Size([1315534, 768]), term_id_pairs: 1315534, canonical mappings: 474316
Found 8401 unique terms in 'active_ingredients_split'


Mapping active_ingredients_split NER to umls: 100%|██████████| 38900/38900 [00:04<00:00, 9470.99it/s] 


Normalization time for 'active_ingredients_split': 0:26:57
Output saved to: /scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full_AP_mapped_active_ingredients.csv


In [34]:
fda_embedded_full = pd.read_csv(output_file)
fda_embedded_full.head()

,active_ingredients_split,linkbert_umls_split,drug_umls_termid,drug_umls_term_norm,drug_umls_closest_3,drug_umls_cdist
0,HYDROCHLOROTHIAZIDE,Hydrochlorothiazide,C0020261,Hydrochlorothiazide,"[['hydrochlorothiazide', 'C0020261'], ['Hydroc...",0.0055
1,LISINOPRIL,Lisinopril,C0065374,Lisinopril,"[['lisinopril', 'C0065374'], ['Lisinopril', 'C...",0.0000
2,CALCITRIOL,Calcitriol,C0006674,Calcitriol,"[['calcitriol', 'C0006674'], ['CALCITRIOL', 'C...",0.0000
3,CARBIDOPA,Carbidopa,C0006982,Carbidopa,"[['Carbidopa', 'C0006982'], ['carbidopa', 'C00...",0.0078
4,LEVODOPA,Levodopa,C0023570,Levodopa,"[['Levodopa', 'C0023570'], ['levodopa', 'C0023...",0.0110


In [35]:
fda_embedded_full.shape

(38900, 6)

In [36]:
fda_embedded_with_details = pd.concat([
    df_orig.reset_index(drop=True), 
    fda_embedded_full[['drug_umls_term_norm', 'drug_umls_termid']].reset_index(drop=True)
], axis=1)
fda_embedded_with_details.head()

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies,active_ingredients_substance_brand,active_ingredients_split,drug_umls_term_norm,drug_umls_termid
0,ANDA076194,WATSON LABS,ORIG,AP,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"Actavis Pharma, Inc.","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[],"HYDROCHLOROTHIAZIDE, LISINOPRIL, LISINOPRIL AN...",HYDROCHLOROTHIAZIDE,Hydrochlorothiazide,C0020261
1,ANDA076194,WATSON LABS,ORIG,AP,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,LISINOPRIL AND HYDROCHLOROTHIAZIDE,"Actavis Pharma, Inc.","HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[],"HYDROCHLOROTHIAZIDE, LISINOPRIL, LISINOPRIL AN...",LISINOPRIL,Lisinopril,C0065374
2,ANDA076206,ROCKWELL MEDCL,ORIG,AP,20030917,NaN,2003,CALCITRIOL,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,NaN,NaN,CALCITRIOL,CALCITRIOL,Calcitriol,C0006674
3,ANDA076212,APOTEX,ORIG,AP,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,"CARBIDOPA, LEVODOPA, CARBIDOPA AND LEVODOPA",CARBIDOPA,Carbidopa,C0006982
4,ANDA076212,APOTEX,ORIG,AP,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,"CARBIDOPA, LEVODOPA, CARBIDOPA AND LEVODOPA",LEVODOPA,Levodopa,C0023570


In [37]:
fda_embedded_with_details.shape

(38900, 27)

In [38]:
fda_embedded_with_details.to_csv("/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_full_AP_mapped_active_ingredients.csv",index=False)

### add parents terminology
from ../04/UMLS_Drug_Parent

In [42]:
fda_embedded_with_details = pd.read_csv("/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_full_AP_mapped_active_ingredients.csv")

In [43]:
import json

with open("../04_normalization/data/parent_mappings/umls_cui_set_full_dataset.json", "r") as f:
    all_umls_ids = set(json.load(f))

with open("../04_normalization/data/parent_mappings/umls_cui2_to_cui1_mapping.json", "r") as f:
    cui2_to_cui1 = json.load(f)

with open("../04_normalization/data/parent_mappings/umls_cui_to_str.json", "r") as f:
    cui_to_str = json.load(f)

with open("../04_normalization/data/parent_mappings/parent_counts.json", "r") as f:
    parent_counts = json.load(f)

In [44]:
def get_all_mappings_with_labels(start_cui, cui2_to_cui1, cui_to_str):
    """
    Return all CUIs reachable from `start_cui` (any depth) and their labels.

    Parameters
    ----------
    start_cui : str
        The root CUI to traverse from (will be included in `all_ids`).
    cui2_to_cui1 : dict[str, list[str]]
        Mapping dictionary where each CUI may map to one or more CUIs.
    cui_to_str : dict[str, str]
        Dictionary mapping CUI IDs to human-readable string labels.

    Returns
    -------
    all_ids : set[str]
        Set of all reachable CUIs, including `start_cui`.
    all_pairs : list[tuple[str, str]]
        List of (CUI, label) pairs for all reachable CUIs,
        **excluding the starting CUI**.
    """
    visited = set()
    stack = [start_cui]

    while stack:
        cui = stack.pop()
        if cui in visited:
            continue
        visited.add(cui)

        if cui in cui2_to_cui1:
            for nxt in cui2_to_cui1[cui]:
                if nxt not in visited:
                    stack.append(nxt)

    # Build list of (cui, label), excluding the start CUI
    all_pairs = [
        (cui, cui_to_str[cui])
        for cui in visited
        if cui != start_cui and cui in cui_to_str
    ]

    return all_pairs


def assign_nearest_dataset_parents(
    df: pd.DataFrame,
    cui2_to_cui1: Dict[str, List[str]],
    cui_to_str: Dict[str, str],
    all_umls_ids: Set[str],
    parent_counts: Dict[str, int],
    id_column: str = "drug_umls_termid",
    tokens_column: str = "drug_umls_term_norm"
) -> pd.DataFrame:

    parent_ids: list[str] = []
    parent_labels: list[str] = []

    mapped_to_parent = defaultdict(set)      # "drug (mapping)" -> list of children

    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Raw split
        raw_ids = str(row[id_column]).split("|")
        raw_tokens = str(row[tokens_column]).split("|")
    
        # Filter based on ID != -1 while keeping alignment
        input_ids = []
        input_tokens = []
        for tid, tok in zip(raw_ids, raw_tokens):
            tid = tid.strip()
            tok = tok.strip()
            if not tid or tid == "-1":
                continue
            input_ids.append(tid)
            input_tokens.append(tok)
    
        row_parents: list[str] = []
        row_labels: list[str] = []
    
        # Loop over aligned ids + tokens
        for child_token, child_id in zip(input_tokens, input_ids):
            parents = get_all_mappings_with_labels(child_id, cui2_to_cui1, cui_to_str)  # list[(parent_id, parent_label)]
            if not parents:
                continue
    
            for parent_id, parent_label in parents:
                # Skip if parent not in counts (defensive) or too many children
                if parent_id not in parent_counts:
                    continue
                if parent_counts[parent_id] > 20:
                    continue
    
                if parent_id in all_umls_ids:
                    row_parents.append(parent_id)
                    row_labels.append(parent_label)
    
                    # Use the *token* for the child, not the ID
                    key = f"{parent_label} ({parent_id})"
                    mapped_to_parent[key].add(child_token)
                        
            
        parent_ids.append("|".join(row_parents) if row_parents else "-1")
        parent_labels.append("|".join(row_labels) if row_labels else "-1")
    
    # Attach results to a copy of the DataFrame
    result = df.copy()
    result["nearest_dataset_parent_umls"] = parent_ids
    result["nearest_dataset_parent_umls_label"] = parent_labels

    return result, mapped_to_parent

In [45]:
fda_embedded_with_details_parent, mapped_to_parent_fda = assign_nearest_dataset_parents(
    fda_embedded_with_details,
    cui2_to_cui1,
    cui_to_str,
    all_umls_ids,
    parent_counts,
    id_column = "drug_umls_termid"
)

In [46]:
fda_embedded_with_details_parent.head()

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies,active_ingredients_substance_brand,active_ingredients_split,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls,nearest_dataset_parent_umls_label
0,ANDA076194,WATSON LABS,ORIG,AP,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,"HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[],"HYDROCHLOROTHIAZIDE, LISINOPRIL, LISINOPRIL AN...",HYDROCHLOROTHIAZIDE,Hydrochlorothiazide,C0020261,-1,-1
1,ANDA076194,WATSON LABS,ORIG,AP,20020701,NaN,2002,LISINOPRIL AND HYDROCHLOROTHIAZIDE,TABLET,ORAL,...,"HYDROCHLOROTHIAZIDE, LISINOPRIL",INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...,[],"HYDROCHLOROTHIAZIDE, LISINOPRIL, LISINOPRIL AN...",LISINOPRIL,Lisinopril,C0065374,-1,-1
2,ANDA076206,ROCKWELL MEDCL,ORIG,AP,20030917,NaN,2003,CALCITRIOL,INJECTABLE,INJECTION,...,NaN,NaN,NaN,NaN,CALCITRIOL,CALCITRIOL,Calcitriol,C0006674,-1,-1
3,ANDA076212,APOTEX,ORIG,AP,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,"CARBIDOPA, LEVODOPA, CARBIDOPA AND LEVODOPA",CARBIDOPA,Carbidopa,C0006982,-1,-1
4,ANDA076212,APOTEX,ORIG,AP,20040616,NaN,2004,CARBIDOPA AND LEVODOPA,"TABLET, EXTENDED RELEASE",ORAL,...,NaN,NaN,NaN,NaN,"CARBIDOPA, LEVODOPA, CARBIDOPA AND LEVODOPA",LEVODOPA,Levodopa,C0023570,-1,-1


In [47]:
# 1) rows that have a valid parent
mask = fda_embedded_with_details_parent["nearest_dataset_parent_umls"].ne("-1") & fda_embedded_with_details_parent["nearest_dataset_parent_umls"].notna()

# 2) make copies of those rows
parent_rows = fda_embedded_with_details_parent.loc[mask].copy()

# 3) overwrite drug fields with parent fields in the copies
parent_rows["drug_umls_termid"] = parent_rows["nearest_dataset_parent_umls"]
parent_rows["drug_umls_term_norm"] = parent_rows["nearest_dataset_parent_umls_label"]
parent_rows = parent_rows.drop(columns=["nearest_dataset_parent_umls", "nearest_dataset_parent_umls_label"])

# 4) append back to original df
fda_embedded_with_details = (
    pd.concat([fda_embedded_with_details, parent_rows], ignore_index=True)
      .drop_duplicates()
)


In [49]:
fda_embedded_with_details[fda_embedded_with_details['application_number']=="ANDA203518"]

,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies,active_ingredients_substance_brand,active_ingredients_split,drug_umls_term_norm,drug_umls_termid
1304,ANDA203518,TRIS PHARMA INC,ORIG,AP,20150512,NaN,2015,MORPHINE SULFATE,SOLUTION,ORAL,...,MORPHINE SULFATE,Bryant Ranch Prepack,MORPHINE SULFATE,1 INDICATIONS AND USAGE Morphine Sulfate Oral ...,1 INDICATIONS AND USAGE Morphine Sulfate Oral ...,[],MORPHINE SULFATE,MORPHINE SULFATE,"Sulfate, Morphine",C0066814
39331,ANDA203518,TRIS PHARMA INC,ORIG,AP,20150512,NaN,2015,MORPHINE SULFATE,SOLUTION,ORAL,...,MORPHINE SULFATE,Bryant Ranch Prepack,MORPHINE SULFATE,1 INDICATIONS AND USAGE Morphine Sulfate Oral ...,1 INDICATIONS AND USAGE Morphine Sulfate Oral ...,[],MORPHINE SULFATE,MORPHINE SULFATE,Morphine,C0026549


In [50]:
fda_embedded_with_details.shape

(49990, 27)

In [51]:
fda_embedded_with_details.to_csv("/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_full_AP_mapped_active_ingredients_with_parent.csv",index=False)

## NDAs vs ANDAs (drug only FDA year)

In [3]:
df_orig = pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_drugs_labels_full.csv")

In [6]:
df_orig[df_orig["active_ingredients"].str.contains("cariprazine", case=False, na=False)]


,application_number,sponsor_name,submission_type,submission_status,submission_status_date,submission_class_code_description,year,brand_name,dosage_form,route,...,openfda_generic_name,openfda_route,openfda_substance_name,label_brand_name,label_generic_name,label_manufacturer_name,label_substance_name,indications_first_sent,indications_and_usage,clinical_studies
21966,NDA204370,ABBVIE,ORIG,AP,20150917,Type 1 - New Molecular Entity,2015,VRAYLAR,CAPSULE,ORAL,...,CARIPRAZINE,ORAL,CARIPRAZINE,Vraylar,CARIPRAZINE,"Allergan, Inc.",CARIPRAZINE,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,[]
25270,ANDA213932,SUN PHARM,ORIG,AP,20220930,NaN,2022,CARIPRAZINE HYDROCHLORIDE,CAPSULE,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25272,ANDA213984,ZYDUS,ORIG,AP,20220909,NaN,2022,CARIPRAZINE HYDROCHLORIDE,CAPSULE,ORAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_orig_ndas = df_orig[df_orig['application_number'].str.startswith("NDA")][['application_number', 'active_ingredients', 'brand_name', 'year', 'sponsor_name', 'submission_class_code_description', 'submission_status','indications_first_sent', 'indications_and_usage']]
df_orig_ndas = df_orig_ndas[df_orig_ndas['submission_status']=='AP']
df_orig_ndas.shape

(5325, 9)

In [47]:
df_orig_ndas["active_ingredients_split"] = (
    df_orig_ndas["active_ingredients"]
    .astype(str)
    .str.split(r",\s*|\|\|")  # Matches comma (+ optional space) OR literal ||
)
df_orig_ndas = df_orig_ndas.explode("active_ingredients_split", ignore_index=True)
df_orig_ndas.shape

(6808, 10)

In [48]:
df_orig_ndas.head()

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,indications_and_usage,active_ingredients_split
0,NDA017446,HEXACHLOROPHENE,PHISO-SCRUB,1977,SANOFI AVENTIS US,Type 5 - New Formulation or New Manufacturer,AP,NaN,NaN,HEXACHLOROPHENE
1,NDA017453,DIAZOXIDE,PROGLYCEM,1976,TEVA BRANDED PHARM,Type 3 - New Dosage Form,AP,INDICATIONS AND USAGE PROGLYCEM is indicated f...,INDICATIONS AND USAGE PROGLYCEM is indicated f...,DIAZOXIDE
2,NDA017476,POTASSIUM CHLORIDE,SLOW-K,1975,NOVARTIS,Type 5 - New Formulation or New Manufacturer,AP,NaN,NaN,POTASSIUM CHLORIDE
3,NDA017481,MEBENDAZOLE,VERMOX,1974,JANSSEN PHARMS,Type 1 - New Molecular Entity,AP,NaN,NaN,MEBENDAZOLE
4,NDA017513,STERILE WATER FOR IRRIGATION,STERILE WATER IN PLASTIC CONTAINER,1974,OTSUKA ICU MEDCL,Type 5 - New Formulation or New Manufacturer,AP,INDICATIONS AND USAGE Sterile Water for Irriga...,INDICATIONS AND USAGE Sterile Water for Irriga...,STERILE WATER FOR IRRIGATION


In [49]:
df_orig_ndas.active_ingredients.nunique(), df_orig_ndas.active_ingredients_split.nunique()

(2500, 2219)

In [50]:
df_orig_andas = df_orig[df_orig['application_number'].str.startswith("ANDA")][['application_number', 'active_ingredients', 'brand_name', 'year', 'sponsor_name', 'submission_class_code_description', 'submission_status','indications_first_sent', 'indications_and_usage']]
df_orig_andas = df_orig_andas[df_orig_andas['submission_status']=='AP']

df_orig_andas.shape

(19450, 9)

In [51]:
df_orig_andas["active_ingredients_split"] = (
    df_orig_andas["active_ingredients"]
    .astype(str)
    .str.split(r",\s*|\|\|")  # Matches comma (+ optional space) OR literal ||
)
df_orig_andas = df_orig_andas.explode("active_ingredients_split", ignore_index=True)
df_orig_andas.shape

(22163, 10)

In [52]:
df_orig_andas.active_ingredients.nunique(), df_orig_andas.active_ingredients_split.nunique()

(1418, 1258)

In [53]:
df_orig_ndas[df_orig_ndas['active_ingredients_split']=="TIGECYCLINE"]

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,indications_and_usage,active_ingredients_split
921,NDA208744,TIGECYCLINE,TIGECYCLINE,2018,ACCORD HLTHCARE INC,Type 5 - New Formulation or New Manufacturer,AP,NaN,NaN,TIGECYCLINE
2336,NDA211158,TIGECYCLINE,TIGECYCLINE,2018,AMNEAL,Type 5 - New Formulation or New Manufacturer,AP,1 INDICATIONS AND USAGE Tigecycline is a tetra...,1 INDICATIONS AND USAGE Tigecycline is a tetra...,TIGECYCLINE
5222,NDA021821,TIGECYCLINE,TYGACIL,2005,PF PRISM CV,Type 1 - New Molecular Entity,AP,1 INDICATIONS AND USAGE TYGACIL is a tetracycl...,1 INDICATIONS AND USAGE TYGACIL is a tetracycl...,TIGECYCLINE
5581,NDA205645,TIGECYCLINE,TIGECYCLINE,2016,FRESENIUS KABI USA,Type 5 - New Formulation or New Manufacturer,AP,1 INDICATIONS AND USAGE Tigecycline for inject...,1 INDICATIONS AND USAGE Tigecycline for inject...,TIGECYCLINE


In [54]:
set1 = set(df_orig_andas['active_ingredients_split'])
set2 = set(df_orig_ndas['active_ingredients_split'])

# Items in set1 but not in set2
difference = set1 - set2
len(difference)

60

In [55]:
# 1. Create sets using tuples (tuples are hashable, lists are not)
set1 = set(df_orig_andas['active_ingredients_split'])
set2 = set(df_orig_ndas['active_ingredients_split'])

# 2. Find the difference
difference = set1 - set2

# 3. Filter the original DataFrame
# We check if the tuple version of the row is in our "difference" set
df_only_in_andas = df_orig_andas[
    df_orig_andas['active_ingredients_split'].isin(difference)
]
df_only_in_andas

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,indications_and_usage,active_ingredients_split
204,ANDA078293,Oxybutynin Chloride,Oxybutynin Chloride,2007,MYLAN PHARMS INC,NaN,AP,NaN,NaN,Oxybutynin Chloride
294,ANDA080188,TESTOSTERONE PROPIONATE,TESTOSTERONE PROPIONATE,1974,WATSON LABS,NaN,AP,NaN,NaN,TESTOSTERONE PROPIONATE
306,ANDA080767,METHYLTESTOSTERONE,METHYLTESTOSTERONE,1974,IMPAX LABS,NaN,AP,INDICATIONS AND USAGE 1. Males Androgens are i...,INDICATIONS AND USAGE 1. Males Androgens are i...,METHYLTESTOSTERONE
330,ANDA083213,"HYDROCORTISONE ACETATE, PRAMOXINE HYDROCHLORIDE",PRAMOSONE,1973,FERNDALE LABS,NaN,AP,NaN,NaN,PRAMOXINE HYDROCHLORIDE
332,ANDA083247,PENTOBARBITAL SODIUM,NEMBUTAL,1982,SCIEGEN PHARMS INC,NaN,AP,NaN,NaN,PENTOBARBITAL SODIUM
...,...,...,...,...,...,...,...,...,...,...
21704,ANDA061655,KANAMYCIN SULFATE,KANTREX,1973,APOTHECON,NaN,AP,NaN,NaN,KANAMYCIN SULFATE
21733,ANDA062683,CEPHRADINE,CEPHRADINE,1987,TEVA,NaN,AP,NaN,NaN,CEPHRADINE
21735,ANDA062693,CEPHRADINE,CEPHRADINE,1987,TEVA,NaN,AP,NaN,NaN,CEPHRADINE
21739,ANDA062859,CEPHRADINE,CEPHRADINE,1988,BARR,NaN,AP,NaN,NaN,CEPHRADINE


In [56]:
overlaps = set1 & set2
len(overlaps), list(overlaps)[:10]

(1198,
 ['DOXYCYCLINE HYCLATE',
  'CALCIUM ACETATE',
  'CROTAMITON',
  'PHENDIMETRAZINE TARTRATE',
  'LOPERAMIDE HYDROCHLORIDE',
  'COLCHICINE',
  'ALLOPURINOL SODIUM',
  'MONTELUKAST SODIUM',
  'MOLINDONE HYDROCHLORIDE',
  'METHYCLOTHIAZIDE'])

In [57]:
list(difference)[:10]

['CEFONICID SODIUM',
 'Oxybutynin Chloride',
 'PENICILLIN G SODIUM',
 'AMPHETAMINE ADIPATE',
 'CALCIUM GLUCEPTATE',
 'SODIUM SUCCINATE',
 'KANAMYCIN SULFATE',
 'AMPICILLIN/AMPICILLIN TRIHYDRATE',
 'ULTRAMICROCRYSTALLINE',
 'MICROSIZE']

In [58]:
df_orig_andas[df_orig_andas['active_ingredients_split']=="DIETHYLSTILBESTROL"]

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,indications_and_usage,active_ingredients_split
4772,ANDA083003,DIETHYLSTILBESTROL,STILBESTROL,1973,TABLICAPS,NaN,AP,NaN,NaN,DIETHYLSTILBESTROL


In [59]:
df_orig_ndas[df_orig_ndas['active_ingredients_split'].str.contains("DIETHYLSTILBESTROL")]

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,indications_and_usage,active_ingredients_split


In [60]:
df_to_process = pd.concat([df_orig_ndas, df_only_in_andas], axis=0, ignore_index=True).drop_duplicates()
df_to_process.shape

(7030, 10)

In [61]:
df_to_process.to_csv("/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP.csv",index=False)

### embed the drug terms

In [24]:
import sys
sys.path.append("../04_normalization")   # adjust path to your real folder
from neural_based_nen import main

In [25]:
data_dir =  "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/"

In [37]:
mapping_type = "drug"
col_to_map = "active_ingredients_split"
input_file = "/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP.csv"
output_file = "/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP_mapped_active_ingredients.csv"

main(mapping_type, col_to_map, data_dir, input_file, output_file, stats_dir=None)

Input file: /scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP.csv


/home/sdonev/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Using terminology: umls
Using distance threshold: 8.2
Starting normalization for: DRUG with cdist 8.2
Loading embeddings from /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/umls/embeddings with prefix UMLS_emb...
Loading term-id pairs from /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/umls/umls_term_id_pairs_combined.json...
Found COMBINED embeddings file, loading that one...
Loaded embeddings: torch.Size([1315534, 768]), term_id_pairs: 1315534, canonical mappings: 474316
Found 2278 unique terms in 'active_ingredients_split'


Mapping active_ingredients_split NER to umls: 100%|██████████| 7030/7030 [00:01<00:00, 6696.56it/s] 

Normalization time for 'active_ingredients_split': 0:07:03
Output saved to: /scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP_mapped_active_ingredients.csv


In [38]:
fda_embedded = pd.read_csv("/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP_mapped_active_ingredients.csv")
fda_embedded.head()

,active_ingredients_split,linkbert_umls_split,drug_umls_termid,drug_umls_term_norm,drug_umls_closest_3,drug_umls_cdist
0,HEXACHLOROPHENE,Hexachlorophene,C0019435,Hexachlorophene,"[['hexachlorophene', 'C0019435'], ['hexachloro...",0.0000
1,DIAZOXIDE,Diazoxide,C0012022,Diazoxide,"[['diazoxide', 'C0012022'], ['Diazoxide', 'C00...",0.0000
2,POTASSIUM CHLORIDE,potassium chloride,C0032825,"Chloride, Potassium","[['potassium chloride', 'C0017725'], ['potassi...",0.0000
3,MEBENDAZOLE,Mebendazole,C0025023,Mebendazole,"[['Mebendazole', 'C0025023'], ['mebendazole', ...",0.0096
4,STERILE WATER FOR IRRIGATION,"WATER FOR IRRIGATION,STERILE",C0590256,"WATER FOR IRRIGATION,STERILE","[['WATER FOR IRRIGATION,STERILE', 'C0590256'],...",3.2935
...,...,...,...,...,...,...
7025,KANAMYCIN SULFATE,Kanamycin Sulfate,C0700540,Kanamycin Sulfate,"[['Kanamycin Sulfate', 'C0700540'], ['kanamyci...",0.0000
7026,CEPHRADINE,Cephradine,C0007738,Cephradine,"[['cephradine', 'C0007738'], ['Cephradine', 'C...",0.0000
7027,CEPHRADINE,Cephradine,C0007738,Cephradine,"[['cephradine', 'C0007738'], ['Cephradine', 'C...",0.0000
7028,CEPHRADINE,Cephradine,C0007738,Cephradine,"[['cephradine', 'C0007738'], ['Cephradine', 'C...",0.0000


In [39]:
fda_embedded[fda_embedded['drug_umls_termid']=="-1"]

,active_ingredients_split,linkbert_umls_split,drug_umls_termid,drug_umls_term_norm,drug_umls_closest_3,drug_umls_cdist
14,TECHNETIUM TC-99M ETIDRONATE KIT,TECHNETIUM TC-99M ETIDRONATE KIT,-1,TECHNETIUM TC-99M ETIDRONATE KIT,"[['technetium Tc 99m (Sn) etidronate', 'C00758...",9.4178
24,TECHNETIUM TC-99M ETIDRONATE KIT,TECHNETIUM TC-99M ETIDRONATE KIT,-1,TECHNETIUM TC-99M ETIDRONATE KIT,"[['technetium Tc 99m (Sn) etidronate', 'C00758...",9.4178
26,GLACIAL,GLACIAL,-1,GLACIAL,"[['arctic ice', 'C0025368'], ['arctic ice', 'C...",12.0375
36,TECHNETIUM TC-99M SUCCIMER KIT,TECHNETIUM TC-99M SUCCIMER KIT,-1,TECHNETIUM TC-99M SUCCIMER KIT,"[['technetium tc-99m succimer', 'C0075928'], [...",8.5834
76,BENTIROMIDE,BENTIROMIDE,-1,BENTIROMIDE,"[['bentazon', 'C0053118'], ['bentazon (substan...",11.3292
...,...,...,...,...,...,...
6970,ULTRAMICROCRYSTALLINE,ULTRAMICROCRYSTALLINE,-1,ULTRAMICROCRYSTALLINE,"[['cellulose, microcrystalline', 'C0072988'], ...",12.6747
7001,AMPICILLIN/AMPICILLIN TRIHYDRATE,AMPICILLIN/AMPICILLIN TRIHYDRATE,-1,AMPICILLIN/AMPICILLIN TRIHYDRATE,"[['ampicillin trihydrate', 'C0700443'], ['ampi...",8.8571
7002,AMPICILLIN/AMPICILLIN TRIHYDRATE,AMPICILLIN/AMPICILLIN TRIHYDRATE,-1,AMPICILLIN/AMPICILLIN TRIHYDRATE,"[['ampicillin trihydrate', 'C0700443'], ['ampi...",8.8571
7003,MICROSIZE,MICROSIZE,-1,MICROSIZE,"[['microplastic', 'C5197738'], ['Microplastics...",11.5207


In [62]:
fda_embedded_with_details = pd.concat([
    df_to_process.reset_index(drop=True), 
    fda_embedded[['drug_umls_term_norm', 'drug_umls_termid']].reset_index(drop=True)
], axis=1)

In [63]:
fda_embedded_with_details.head()

,application_number,active_ingredients,brand_name,year,sponsor_name,submission_class_code_description,submission_status,indications_first_sent,indications_and_usage,active_ingredients_split,drug_umls_term_norm,drug_umls_termid
0,NDA017446,HEXACHLOROPHENE,PHISO-SCRUB,1977,SANOFI AVENTIS US,Type 5 - New Formulation or New Manufacturer,AP,NaN,NaN,HEXACHLOROPHENE,Hexachlorophene,C0019435
1,NDA017453,DIAZOXIDE,PROGLYCEM,1976,TEVA BRANDED PHARM,Type 3 - New Dosage Form,AP,INDICATIONS AND USAGE PROGLYCEM is indicated f...,INDICATIONS AND USAGE PROGLYCEM is indicated f...,DIAZOXIDE,Diazoxide,C0012022
2,NDA017476,POTASSIUM CHLORIDE,SLOW-K,1975,NOVARTIS,Type 5 - New Formulation or New Manufacturer,AP,NaN,NaN,POTASSIUM CHLORIDE,"Chloride, Potassium",C0032825
3,NDA017481,MEBENDAZOLE,VERMOX,1974,JANSSEN PHARMS,Type 1 - New Molecular Entity,AP,NaN,NaN,MEBENDAZOLE,Mebendazole,C0025023
4,NDA017513,STERILE WATER FOR IRRIGATION,STERILE WATER IN PLASTIC CONTAINER,1974,OTSUKA ICU MEDCL,Type 5 - New Formulation or New Manufacturer,AP,INDICATIONS AND USAGE Sterile Water for Irriga...,INDICATIONS AND USAGE Sterile Water for Irriga...,STERILE WATER FOR IRRIGATION,"WATER FOR IRRIGATION,STERILE",C0590256
...,...,...,...,...,...,...,...,...,...,...,...,...
7025,ANDA061655,KANAMYCIN SULFATE,KANTREX,1973,APOTHECON,NaN,AP,NaN,NaN,KANAMYCIN SULFATE,Kanamycin Sulfate,C0700540
7026,ANDA062683,CEPHRADINE,CEPHRADINE,1987,TEVA,NaN,AP,NaN,NaN,CEPHRADINE,Cephradine,C0007738
7027,ANDA062693,CEPHRADINE,CEPHRADINE,1987,TEVA,NaN,AP,NaN,NaN,CEPHRADINE,Cephradine,C0007738
7028,ANDA062859,CEPHRADINE,CEPHRADINE,1988,BARR,NaN,AP,NaN,NaN,CEPHRADINE,Cephradine,C0007738


In [64]:
fda_embedded_with_details.to_csv("/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP_embedded_drug.csv",index=False)

In [43]:
result = fda_embedded_with_details.groupby("drug_umls_term_norm", as_index=False).agg(
    application_numbers=("application_number", list),
    brand_names=("brand_name", list),
    all_years=("year", list),
    first_approval_year=("year", "min"),
    sponsors=("sponsor_name", list)
)
result.head()

,drug_umls_term_norm,application_numbers,brand_names,all_years,first_approval_year,sponsors
0,"3,5-dihydroxy-4-isopropylstilbene",[NDA215272],[VTAMA],[2022],2022,[ORGANON LLC]
1,"4-(alpha-(p-(diethylamino)phenyl)-(2,5-disulfo...",[NDA018310],[LYMPHAZURIN],[1981],1981,[COVIDIEN]
2,"5'-Phosphate, Riboflavin","[NDA203324, NDA219910]","[PHOTREXA VISCOUS IN DEXTRAN 20%, EPIOXA HD/EP...","[2016, 2025]",2016,"[GLAUKOS, GLAUKOS]"
3,6 Aminocaproic Acid,"[NDA018590, NDA015197, NDA015230, NDA015229]","[AMINOCAPROIC ACID, AMICAR, AMICAR, AMICAR]","[1982, 1964, 1964, 1964]",1964,"[BAXTER HLTHCARE, HIKMA, HIKMA, EPIC PHARMA LLC]"
4,ACETOHEXAMIDE,[NDA013378],[DYMELOR],[1964],1964,[LILLY]
...,...,...,...,...,...,...
2253,zileuton,"[NDA020471, NDA022052]","[ZYFLO, ZYFLO CR]","[1996, 2007]",1996,"[CHIESI, CHIESI]"
2254,zilucoplan sodium,[NDA216834],[ZILBRYSQ],[2023],2023,[UCB INC]
2255,zinc chloride,[NDA018959],[ZINC CHLORIDE IN PLASTIC CONTAINER],[1986],1986,[HOSPIRA]
2256,zolmitriptan,"[NDA020768, NDA021231, NDA021450]","[ZOMIG, ZOMIG-ZMT, ZOMIG]","[1997, 2001, 2003]",1997,"[IPR, ASTRAZENECA, AMNEAL]"


In [44]:
result.to_csv("/scratch/sdonev/Preclinical_Pipeline/10_use_case_disease_focus/out/FDA_nda_anda_AP_summary_over_mapped_drugs.csv",index=False)